## Cleaning

In [23]:
import pandas as pd
import numpy as np


def calculate_metrics(file_path):

    # Row 4 contains the actual column headers
    df = pd.read_excel(file_path, header=4)

    # Remove accidental spaces from column names
    df.columns = df.columns.astype(str).str.strip()

    # Keep required columns
    df = df[["Date", "Net Asset Value", "Plan", "Option"]]

    # Rename
    df.columns = ["Date", "NAV", "Plan", "Option"]

    # Convert data types
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["NAV"] = pd.to_numeric(df["NAV"], errors="coerce")

    # Keep direct plans and growth options despite label variations
    df = df[
        df["Plan"].astype(str).str.contains("direct", case=False, na=False) &
        df["Option"].astype(str).str.contains("growth", case=False, na=False)
    ]

    # Study period: 01-Jan-2022 to 31-Dec-2025
    df = df[
        (df["Date"] >= "2022-01-01") &
        (df["Date"] <= "2025-12-31")
    ]

    # Remove missing values
    df = df.dropna(subset=["Date", "NAV"])

    # Sort by date
    df = df.sort_values("Date").reset_index(drop=True)

    if df.empty:
        raise ValueError(f"No usable Direct Growth rows found in {file_path}")

    # Daily returns
    df["Daily_Return"] = df["NAV"].pct_change()

    # CAGR
    start_nav = df["NAV"].iloc[0]
    end_nav = df["NAV"].iloc[-1]

    years = (
        df["Date"].iloc[-1] - df["Date"].iloc[0]
    ).days / 365.25

    cagr = (end_nav / start_nav) ** (1 / years) - 1

    # Annualized volatility
    volatility = df["Daily_Return"].std() * np.sqrt(252)

    # Maximum drawdown
    df["Running_Max"] = df["NAV"].cummax()

    df["Drawdown"] = (
        (df["NAV"] - df["Running_Max"]) /
        df["Running_Max"]
    )

    max_drawdown = df["Drawdown"].min()

    return df, {
        "Start NAV": start_nav,
        "End NAV": end_nav,
        "CAGR": cagr,
        "Volatility": volatility,
        "Maximum Drawdown": max_drawdown
    }

In [24]:
df = pd.read_excel("Large_cap/nippon.xlsx", header=None)

print(df.head(10).to_string())

                                                                       0                 1           2                    3            4              5
0                Historical NAV Data for From 03-Jan-2022 to 31-Dec-2025               NaN         NaN                  NaN          NaN            NaN
1                                               Nippon India Mutual Fund               NaN         NaN                  NaN          NaN            NaN
2                                            Nippon India Large Cap Fund               NaN         NaN                  NaN          NaN            NaN
3  Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option               NaN         NaN                  NaN          NaN            NaN
4                                                        Net Asset Value  Repurchase Price  Sale Price                 Date         Plan         Option
5                                                                54.3991               N

In [26]:
from pathlib import Path

funds = {
    "Nippon India Large Cap": "Large_cap/nippon.xlsx",
    "ICICI Prudential Large Cap": "Large_cap/ICICI.xlsx",
    "HDFC Large Cap": "Large_cap/hdfc.xlsx",
    "DSP Large Cap": "Large_cap/dsp.xlsx",
    "Bandhan Large Cap": "Large_cap/bandhan.xlsx"
}

output_dir = Path("cleaned_data")
output_dir.mkdir(exist_ok=True)

results = []

for fund_name, file in funds.items():

    df, metrics = calculate_metrics(file)

    output_file = output_dir / f"{Path(file).stem}_cleaned.csv"
    df.to_csv(output_file, index=False)

    metrics["Fund"] = fund_name
    metrics["Cleaned File"] = str(output_file)
    results.append(metrics)

results_df = pd.DataFrame(results)

print(results_df)
print(f"Saved {len(results)} cleaned datasets to {output_dir}/")

   Start NAV    End NAV      CAGR  Volatility  Maximum Drawdown  \
0    54.3991   105.3421  0.180049    0.136721         -0.153680   
1    71.0800   127.1700  0.156883    0.123998         -0.153933   
2   732.7530  1270.3590  0.149111    0.131937         -0.163646   
3   312.6130   534.9380  0.144048    0.124441         -0.176848   
4    56.4500    92.7700  0.132523    0.135343         -0.194181   

                         Fund                      Cleaned File  
0      Nippon India Large Cap   cleaned_data/nippon_cleaned.csv  
1  ICICI Prudential Large Cap    cleaned_data/ICICI_cleaned.csv  
2              HDFC Large Cap     cleaned_data/hdfc_cleaned.csv  
3               DSP Large Cap      cleaned_data/dsp_cleaned.csv  
4           Bandhan Large Cap  cleaned_data/bandhan_cleaned.csv  
Saved 5 cleaned datasets to cleaned_data/
